In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# <span style="color:red">ch9.03_ChatOpenAI과 렝체인을 활용한 검증</span>

# 1. 환경(패키지 및 환경변수)

- pip install openai langchain-openai
- pip install langchain-upstage

In [1]:
%pip install langchain-upstage # -q 메세지 안나오게하는 설정


  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/558.8 kB ? eta -:--:--
   ---------------------------------------- 558.8/558.8 kB 6.3 MB/s eta 0:00:00

   ---------------------------------------- 0/7 [pypdf]
   ---------------------------------------- 0/7 [pypdf]
   ---------------------------------------- 0/7 [pypdf]
  Attempting uninstall: packaging
   ---------------------------------------- 0/7 [pypdf]
    Found existing installation: packaging 25.0
   ---------------------------------------- 0/7 [pypdf]
    Uninstalling packaging-25.0:
   ---------------------------------------- 0/7 [pypdf]
      Successfully uninstalled packaging-25.0
   ---------------------------------------- 0/7 [pypdf]
   ----- ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# 2. LLM 답변 생성

## 2.1 OpenAi SDK를 사용

In [2]:
import os
upstage_api_key = os.getenv('UPSTAGE_API_KEY')

In [3]:
from openai import OpenAI 
 
client = OpenAI(
    api_key=upstage_api_key,
    base_url="https://api.upstage.ai/v1"
)
 
stream = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "2020년 월드시리즈 누가 우승했어?"
        }
    ],
    stream=False,
)

In [ ]:
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")
 
# Use with stream=False
# print(stream.choices[0].message.content)

In [4]:
print(stream.choices[0].message.content)

2020년 월드시리즈에서 **로스앤젤레스 다저스(Los Angeles Dodgers)**가 우승했습니다.  

- **결승 상대**: 탬파베이 레이스(Tampa Bay Rays)  
- **시리즈 전적**: 4승 2패로 다저스 승리  
- **MVP**: 다저스의 불펜 투수 **코리 시거(Corey Seager)**가 시리즈 MVP로 선정되었습니다.  

이 우승으로 다저스는 1988년 이후 32년 만에 통산 7번째 월드시리즈 정상에 올랐으며, 2020년 시즌은 코로나19 팬데믹의 영향으로 중립 구장(글래스고 돔)에서 개최되었습니다.  

참고로, 2020년은 월드시리즈가 연장전 없이 7전 4선승제로 진행된 마지막 해였으며, 2022년부터 다시 정규 일정으로 복귀했습니다.


# 2.2 Langchain을 선택

- 발급받은 API Key를 .env에 UPSTAGE_API_KEY라고 저장하면 별도의 설정없이 ChatUpstage를 바로 사용

In [7]:
from langchain_upstage import ChatUpstage
from langchain_core.messages import SystemMessage, HumanMessage
llm = ChatUpstage() 
messages = [
    SystemMessage(content="너는 친절하게 대답해주는 비서야"),
    HumanMessage(content="2020년 월드시리즈는 누가 우승했어?")
]
ai_message = llm.invoke(input=messages)
ai_message

AIMessage(content='2020년 월드시리즈는 로스앤젤레스 다저스가 우승했습니다. 이는 1988년 이후 32년 만의 우승이었으며, 특히 2020년 시즌은 COVID-19로 인해 경기 방식이 변경되어 더 특별한 의미를 가졌습니다. 다저스는 템파베이 레이스를 시리즈 4승 2패로 꺾고 우승을 차지했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 37, 'total_tokens': 123, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'solar-mini-250422', 'system_fingerprint': None, 'id': 'bc3bf56e-cd9d-4eaa-9a71-634e7880a4d5', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--f3ed0fcf-26f2-4d81-bddf-59f021b68e4d-0', usage_metadata={'input_tokens': 37, 'output_tokens': 86, 'total_tokens': 123, 'input_token_details': {}, 'output_token_details': {}})

In [8]:
ai_message.content

'2020년 월드시리즈는 로스앤젤레스 다저스가 우승했습니다. 이는 1988년 이후 32년 만의 우승이었으며, 특히 2020년 시즌은 COVID-19로 인해 경기 방식이 변경되어 더 특별한 의미를 가졌습니다. 다저스는 템파베이 레이스를 시리즈 4승 2패로 꺾고 우승을 차지했습니다.'